# ライブラリのインポート

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_PATH = "microsoft/deberta-v3-large"  # ヒント: HuggingFace上のDeBERTa v3-largeのモデルID
    MODEL_NAME = "deberta-v3-large"
    MAX_LENGTH = 2048  # ヒント: トークン列の最大長（2のべき乗が一般的）
    EPOCHS = 3  # ヒント: 学習エポック数
    STEPS = 20
    WARMUP_RATIO = 0.05
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-5  # ヒント: fine-tuningの典型的な学習率（2e-5など）
    OPTIM = "adamw_torch"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 3  # ヒント: 交差検証のfold数
    TARGET_COL = "Recommended IND"  # ヒント: 目的変数のカラム名（"Recommended IND"）
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"  # ヒント: 評価指標名（"auc"）
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
    OUTPUT_DIR = "./model"

# 乱数の設定

In [ ]:
def seed_everything(seed):
    
    # YOUR CODE HERE
    # ヒント: random, numpy, os環境変数(PYTHONHASHSEED), torch, CUDA のシードを設定する
    # CFG.USE_GPU が True の場合は torch.backends.cudnn も設定する
    pass

seed_everything(CFG.SEED)

# デバイスの設定

In [ ]:
device = torch.device("cuda" if CFG.USE_GPU else "cpu")
print(device)

# データの読み込み

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

# データの加工

In [ ]:
def make_text_column(df):
    # YOUR CODE HERE
    # ヒント: "Title" と "Review Text" を空白で結合して "text" 列を作る
    pass

def preprocessing(df, clothing_master_df):
    # YOUR CODE HERE
    # ヒント:
    # 1. "Title" と "Review Text" の欠損値を空文字列で埋める
    # 2. clothing_master_df を "Clothing ID" キーで左結合する
    # 3. make_text_column() を呼んで text 列を作る
    pass

In [ ]:
train_df = preprocessing(train_df, clothing_master_df)
test_df = preprocessing(test_df, clothing_master_df)
train_df["labels"] = ___  # ヒント: TARGET_COL を int32 型に変換する

# トークナイザの設定

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH)

def tokenize(sample, tokenizer):
    # YOUR CODE HERE
    # ヒント: tokenizer を使って sample["text"] をトークナイズする
    # max_length と truncation を設定すること
    pass

# データセットの作成

In [ ]:
def mk_dataset(df, tokenizer):
    # YOUR CODE HERE
    # ヒント:
    # 1. Dataset.from_pandas() で DataFrame を HuggingFace Dataset に変換する
    # 2. .map() で tokenize 関数を適用する（fn_kwargs でtokenizer を渡す）
    # 3. .remove_columns(["text"]) で元のテキスト列を削除する
    pass

In [ ]:
ds = mk_dataset(train_df[["text", "labels"]], tokenizer)
ds_test = mk_dataset(test_df[["text"]], tokenizer)

In [ ]:
print(ds)

In [ ]:
print(ds["input_ids"][0])

# モデルの学習と推論

In [ ]:
def compute_metrics(p):
    # YOUR CODE HERE
    # ヒント:
    # 1. p を (label_preds, labels) にアンパックする
    # 2. label_preds に softmax を適用してクラス1の確率を取り出す
    #    （torch.softmax → .numpy()[:, 1]）
    # 3. roc_auc_score(labels, prob_preds) を計算して {"auc": score} を返す
    pass

In [ ]:
def get_args(CFG, fold):
    args = TrainingArguments(
        output_dir=f"./model/{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy=___,  # ヒント: "steps"
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=___,  # ヒント: CFG.METRIC
        greater_is_better=___,  # ヒント: CFG.METRIC_MAXIMIZE_FLAG
        optim=CFG.OPTIM,
        learning_rate=___,  # ヒント: CFG.LEARNING_RATE
        lr_scheduler_type="linear",
        warmup_ratio=CFG.WARMUP_RATIO,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=___,  # ヒント: CFG.EPOCHS
        fp16=___,  # ヒント: GPU使用時にTrue（半精度学習）
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args

In [ ]:
predictions = np.zeros(len(train_df))
test_predictions = np.zeros(len(test_df))

# 交差検証のために、目的変数が各 fold で均一になるようにデータを分割
kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)

for fold, (train_index, valid_index) in enumerate(kfold.split(___, ___)):  # ヒント: df と層化対象カラムを渡す
    # モデルの読み込み & 設定
    config = AutoConfig.from_pretrained(___)  # ヒント: CFG.MODEL_PATH
    config.num_labels = CFG.TARGET_COL_CLASS_NUM
    model = AutoModelForSequenceClassification.from_pretrained(___, config=config)  # ヒント: CFG.MODEL_PATH

    trainer = Trainer(
        model=___,
        args=get_args(CFG, fold),
        train_dataset=ds.select(___),  # ヒント: train_index
        eval_dataset=ds.select(___),   # ヒント: valid_index
        data_collator=DataCollatorWithPadding(tokenizer),
        tokenizer=tokenizer,
        compute_metrics=___,
    )

    # 学習開始
    trainer.train()

    # モデルなどの保存
    trainer.save_model(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")
    tokenizer.save_pretrained(f"{CFG.PREFIX}/{CFG.PREFIX}-fold{fold}")

    # 検証セットの推論
    valid_preds = trainer.predict(ds.select(valid_index)).predictions
    predictions[valid_index] = ___  # ヒント: softmax → numpy()[:, 1]

    # テストセットの推論
    test_preds = trainer.predict(ds_test).predictions
    test_predictions += ___  # ヒント: softmax → numpy()[:, 1] を fold 数で割って加算

In [ ]:
print("CV Score: ", roc_auc_score(train_df[CFG.TARGET_COL], predictions))

In [ ]:
train_df[f"{CFG.PREFIX}_pred_prob"] = predictions
train_df[[f"{CFG.PREFIX}_pred_prob"]].to_csv(f"{CFG.PREFIX}_train_preds.csv", index=False)
test_df[f"{CFG.PREFIX}_pred_prob"] = test_predictions
test_df[[f"{CFG.PREFIX}_pred_prob"]].to_csv(f"{CFG.PREFIX}_test_preds.csv", index=False)

In [ ]:
test_df["target"] = test_df[f"{CFG.PREFIX}_pred_prob"]
test_df[["target"]].to_csv(f"{CFG.PREFIX}_submission.csv", index=False)